# Results — Influencer Attractiveness and AI Disclosure

Revised survey analysis from TUM's Eye Tracking and Decision Making coursework.
Group authors are listed in the README. These notebooks analyze survey responses.

Outputs are cleared for public distribution. Execute with authorized local data.
Original results and narrative conclusions require recalculation after the corrections.
The first three data rows are removed following the original notebook's convention
(two Qualtrics metadata rows and one presumed dummy response). Verify the third-row
exclusion against the original export before interpreting results.

Missing scale items produce missing composite scores. Reliability uses complete
item responses. Regression inference uses HC3 standard errors. Mediation estimates
describe statistical indirect associations and require additional assumptions for
a causal interpretation. Bootstrap intervals are evaluated directly, independently
of whether a total or direct effect is statistically significant.


# summary results

# Import Data

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import statsmodels.api as sm
from scipy.stats import mannwhitneyu
from scipy.stats import pearsonr
import seaborn as sns
import matplotlib.pyplot as plt
import pingouin as pg

In [ ]:
data = pd.read_csv('data/Survey1.csv')

# Data Cleaning

In [ ]:
data_cleaned = data.copy()

In [ ]:
# Remove dummy and first two rows
data_cleaned = data_cleaned.drop([0,1,2])
print(len(data_cleaned))

In [ ]:
ages = pd.to_numeric(data_cleaned["Q7"], errors="coerce")
print(ages.describe())
# Final-sample demographics are reported after screening below.


In [ ]:
# Count occurrences of each unique value in FL_3_DO
fl_3_do_counts = data_cleaned["FL_3_DO"].value_counts()

# Display results
print(fl_3_do_counts)


In [ ]:
# check if remaining participants finished the survey
data_cleaned['Progress'] = pd.to_numeric(data_cleaned['Progress'], errors='coerce')
data_cleaned = data_cleaned.drop(data_cleaned[data_cleaned['Progress'] != 100].index)
print(len(data_cleaned))

In [ ]:
attention = pd.to_numeric(data_cleaned["Q13_3"], errors="coerce")
data_cleaned = data_cleaned.loc[attention.eq(1)].copy()
print("Completed surveys passing the attention check:", len(data_cleaned))


# Data Analysis

In [ ]:
# check mean and median for duration for survey
data_cleaned['Duration (in seconds)'] = pd.to_numeric(data_cleaned['Duration (in seconds)'])
mean_duration = data_cleaned['Duration (in seconds)'].mean()/60
print(f"mean duration (in minutes)': {mean_duration}")
median_duration = data_cleaned['Duration (in seconds)'].median()/60
print(f"median duration (in minutes)': {median_duration}")

## Manipulation Check Attractiveness

In [ ]:
valid_conditions = {
    "attractive_no_ai", "attractive_ai",
    "unattractive_no_ai", "unattractive_ai"
}
data_cleaned["FL_3_DO"] = data_cleaned["FL_3_DO"].astype("string").str.strip()
invalid = ~data_cleaned["FL_3_DO"].isin(valid_conditions)
print("Excluded missing or unknown conditions:", int(invalid.sum()))
data_cleaned = data_cleaned.loc[~invalid].copy()
data_cleaned["attractive"] = data_cleaned["FL_3_DO"].isin(
    ["attractive_no_ai", "attractive_ai"]
)


In [ ]:
# Konvertiere die Spalten in numerische Werte, erzwinge dabei NaN für nicht-numerische Werte
# Q9 Contains items around attractiveness (7 step Likert Scale, 1: Strongly Disagree, ..., 7: Strongly Agree)
data_cleaned[["Q9_1", "Q9_2", "Q9_3"]] = data_cleaned[["Q9_1", "Q9_2", "Q9_3"]].apply(pd.to_numeric, errors='coerce')
data_cleaned["Q9_mean"] = data_cleaned[["Q9_1", "Q9_2", "Q9_3"]].mean(axis=1, skipna=False).apply(pd.to_numeric, errors='coerce')
data_cleaned["Q9_stdev"] = np.std(data_cleaned[["Q9_1", "Q9_2", "Q9_3"]],axis=1, ddof=1)

# Jetzt gruppiere und berechne den Durchschnitt
result_mean_attractive = data_cleaned.groupby("attractive")["Q9_mean"].mean()
result_stdev_attractive = data_cleaned.groupby("attractive")["Q9_mean"].std(ddof=1)

# Zeige die Ergebnisse
print("attractive mean:")
print(result_mean_attractive)
print("")
print("attractive stdev:")
print(result_stdev_attractive)

In [ ]:
# Extrahiere die Werte für jede Gruppe in der 'attractive'-Spalte
group_true = data_cleaned[data_cleaned["attractive"] == True]["Q9_mean"]
group_false = data_cleaned[data_cleaned["attractive"] == False]["Q9_mean"]

# Führe einen t-Test durch
t_stat, p_value = stats.ttest_ind(group_true.dropna(), group_false.dropna(), equal_var=False)

# Zeige das Ergebnis
print(f"T-Statistik: {t_stat}")
print(f"P-Wert: {p_value}")

# Überprüfe, ob der Unterschied signifikant ist (z.B. bei einem Signifikanzniveau von 0.05)
if p_value < 0.05:
    print("Der Unterschied zwischen den Gruppen ist signifikant.")
else:
    print("Der Unterschied zwischen den Gruppen ist nicht signifikant.")

## Manipulation Check: AI IV

In [ ]:
# Group all respondents by whether they saw the picture with an AI label or not
data_cleaned["ai"] = (data_cleaned["FL_3_DO"] == "attractive_ai") | (data_cleaned["FL_3_DO"] == "unattractive_ai")

In [ ]:
# 	Q10 Do you think the person in the ad is generated by AI? (7-step-Likert Scale, 1: Strongly Disagree, ..., 7: Strongly Agree)
# Konvertiere die Spalten in numerische Werte, erzwinge dabei NaN für nicht-numerische Werte
data_cleaned[["Q10"]] = data_cleaned[["Q10"]].apply(pd.to_numeric, errors='coerce')

# Jetzt gruppiere und berechne den Durchschnitt
result_mean = data_cleaned.groupby("ai")["Q10"].mean()
result_stdev = data_cleaned.groupby("ai")["Q10"].std(ddof=1)

# Zeige die Ergebnisse
print("mean:")
print(result_mean)
print("")
print("stdev:")
print(result_stdev)# Extrahiere die Werte für jede Gruppe in der 'attractive'-Spalte
group_true = data_cleaned[data_cleaned["ai"] == True]["Q10"]
group_false = data_cleaned[data_cleaned["ai"] == False]["Q10"]

# Führe einen t-Test durch
t_stat, p_value = stats.ttest_ind(group_true.dropna(), group_false.dropna(), equal_var=False)

# Zeige das Ergebnis
print(f"T-Statistik: {t_stat}")
print(f"P-Wert: {p_value}")

# Überprüfe, ob der Unterschied signifikant ist (z.B. bei einem Signifikanzniveau von 0.05)
if p_value < 0.05:
    print("Der Unterschied zwischen den Gruppen ist signifikant.")
else:
    print("Der Unterschied zwischen den Gruppen ist nicht signifikant.")

In [ ]:
# Q11 Was there an AI label in the ad you viewed? (0: No, 1: Yes)
# Konvertiere die Spalten in numerische Werte, erzwinge dabei NaN für nicht-numerische Werte
data_cleaned[["Q11"]] = data_cleaned[["Q11"]].apply(pd.to_numeric, errors='coerce')

# Jetzt gruppiere und berechne den Durchschnitt
result_mean = data_cleaned.groupby("ai")["Q11"].mean()
result_stdev = data_cleaned.groupby("ai")["Q11"].std(ddof=1)

# Zeige die Ergebnisse
print("mean:")
print(result_mean)
print("")
print("stdev:")
print(result_stdev)

# Extrahiere die Werte für jede Gruppe in der 'attractive'-Spalte
group_true = data_cleaned[data_cleaned["ai"] == True]["Q11"]
group_false = data_cleaned[data_cleaned["ai"] == False]["Q11"]

# Führe einen t-Test durch
t_stat, p_value = stats.ttest_ind(group_true.dropna(), group_false.dropna(), equal_var=False)

# Zeige das Ergebnis
print(f"T-Statistik: {t_stat}")
print(f"P-Wert: {p_value}")

# Überprüfe, ob der Unterschied signifikant ist (z.B. bei einem Signifikanzniveau von 0.05)
if p_value < 0.05:
    print("Der Unterschied zwischen den Gruppen ist signifikant.")
else:
    print("Der Unterschied zwischen den Gruppen ist nicht signifikant.")

In [ ]:
# Primary analyses retain participants regardless of label recognition.
recognized = data_cleaned["Q11"].eq(data_cleaned["ai"].astype(int))
recognition_sensitivity = data_cleaned.loc[recognized].copy()
print("Primary eligible sample:", len(data_cleaned))
print("Label-recognition sensitivity sample:", len(recognition_sensitivity))


In [ ]:
matrix = pd.crosstab(data_cleaned["ai"], data_cleaned["attractive"], margins=True)

# Ausgabe
print(matrix)

## Cronbachs alphas

In [ ]:
def cronbach_alpha(data):
    complete = data.apply(pd.to_numeric, errors="coerce").dropna()
    if len(complete) < 2 or complete.shape[1] < 2:
        return np.nan
    total_variance = complete.sum(axis=1).var(ddof=1)
    if total_variance <= 0:
        return np.nan
    k = complete.shape[1]
    return k / (k - 1) * (1 - complete.var(ddof=1).sum() / total_variance)


In [ ]:
# extract relevant data (start from row 2)
# Q9 Attractiveness related questions
data_9 = data_cleaned[["Q9_1", "Q9_2", "Q9_3"]].apply(pd.to_numeric, errors='coerce')
# Q13 Credibility related questions
data_13 = data_cleaned[["Q13_1", "Q13_2", "Q13_4"]].apply(pd.to_numeric, errors='coerce')
# Q14 Attitude towards the influencer related questions
data_14 = data_cleaned[["Q14_1", "Q14_2", "Q14_3"]].apply(pd.to_numeric, errors='coerce')
# Q15 Product interest related questions
data_15 = data_cleaned[["Q15_1", "Q15_2", "Q15_3"]].apply(pd.to_numeric, errors='coerce')

# Calculate Cronbach's Alpha for the columns
alpha_q9 = round(cronbach_alpha(data_9), 2)
print(f"Cronbach's Alpha for Q9 (attractiveness): {alpha_q9}")

alpha_q13 = round(cronbach_alpha(data_13), 2)
print(f"Cronbach's Alpha for Q13 (credibility): {alpha_q13}")

alpha_q14 = round(cronbach_alpha(data_14), 2)
print(f"Cronbach's Alpha for Q14 (attitude towards the influencer): {alpha_q14}")

alpha_q15 = round(cronbach_alpha(data_15), 2)
print(f"Cronbach's Alpha for Q15 (product interest): {alpha_q15}")

## Survey demographics

In [ ]:
# Konvertiere die Spalte 'Q7' (How old are you?) in numerische Werte
data_cleaned['Q7'] = pd.to_numeric(data_cleaned['Q7'], errors='coerce')

# Entferne Zeilen mit NaN-Werten in 'Q7' oder 'Q8' (What is your gender?, 1: Male, 2: Female, 3: Non-Binary, 4: Other), da sie für die Berechnung nicht relevant sind
data_cleaned_filtered = data_cleaned.dropna(subset=['Q7', 'Q8'])

# Berechne den Altersdurchschnitt pro Geschlecht
gender_age_table = data_cleaned_filtered.groupby('Q8')['Q7'].mean().round(2).reset_index()
gender_age_table.columns = ['gender', 'mean_age']

# Füge die Anzahl (Summe) pro Geschlecht hinzu
gender_counts = data_cleaned_filtered['Q8'].value_counts()
gender_age_table['count'] = gender_age_table['gender'].map(gender_counts)

# Berechne den Gesamtanteil (%) pro Geschlecht
total_count = gender_counts.sum()
gender_age_table['percentage'] = ((gender_age_table['count'] / total_count) * 100).round(2)

# Berechne den Gesamtdurchschnitt des Alters
gesamt_mean = round(data_cleaned_filtered['Q7'].mean(), 2)

# Füge eine Zeile mit den Gesamtwerten hinzu
gender_age_table.loc[len(gender_age_table)] = [
    'Total',
    gesamt_mean,
    total_count,
    100.00
]

print(gender_age_table)

## Variable Correlation

In [ ]:
# Attractivenes
data_cleaned["attractive"] = data_cleaned["attractive"].astype(int)

# AI-Label
data_cleaned["ai"] = data_cleaned["ai"].astype(int)

# Attitude
data_cleaned[["Q13_1", "Q13_2", "Q13_4"]] = data_cleaned[["Q13_1", "Q13_2", "Q13_4"]].apply(pd.to_numeric, errors='coerce')
data_cleaned["credebility"] = data_cleaned[["Q13_1", "Q13_2", "Q13_4"]].mean(axis=1, skipna=False)

# Attractiveness
data_cleaned[["Q14_1", "Q14_2", "Q14_3"]] = data_cleaned[["Q14_1", "Q14_2", "Q14_3"]].apply(pd.to_numeric, errors='coerce')
data_cleaned["attitude"] = data_cleaned[["Q14_1", "Q14_2", "Q14_3"]].mean(axis=1, skipna=False)

# Purchase Intention
data_cleaned[["Q15_1", "Q15_2", "Q15_3"]] = data_cleaned[["Q15_1", "Q15_2", "Q15_3"]].apply(pd.to_numeric, errors='coerce')
data_cleaned["purchase"] = data_cleaned[["Q15_1", "Q15_2", "Q15_3"]].mean(axis=1, skipna=False)


In [ ]:
# choose relevant columns
columns_of_interest = ["attitude", "purchase", "credebility", "attractive", "ai"]
data_subset = data_cleaned[columns_of_interest]

# calc corr
correlation_matrix = data_subset.corr()

# function to calc p-values
def calculate_p_values(df):
    p_values = pd.DataFrame(np.zeros((df.shape[1], df.shape[1])), columns=df.columns, index=df.columns)
    for row in df.columns:
        for col in df.columns:
            if row == col:
                p_values.loc[row, col] = np.nan  # Keine Korrelation mit sich selbst
            else:
                _, p_value = pearsonr(*df[[row, col]].dropna().to_numpy().T)
                p_values.loc[row, col] = p_value
    return p_values

# calc p-values
p_values_matrix = calculate_p_values(data_subset)

# add * if significant effect
def add_significance_stars(corr_matrix, p_values):
    significance_matrix = corr_matrix.astype(object).copy()
    for row in corr_matrix.columns:
        for col in corr_matrix.columns:
            if row != col:
                corr = corr_matrix.loc[row, col]
                p_val = p_values.loc[row, col]
                if p_val < 0.05:
                    significance_matrix.loc[row, col] = f"{corr:.2f}*"
                else:
                    significance_matrix.loc[row, col] = f"{corr:.2f}"
            else:
                significance_matrix.loc[row, col] = ""
    return significance_matrix

# create table with corr
annotated_corr_matrix = add_significance_stars(correlation_matrix, p_values_matrix)

# Maske für die obere Hälfte der Matrix erstellen
mask = np.triu(np.ones_like(correlation_matrix, dtype=bool))

# Heatmap plotten mit einem Rot-Farbschema
plt.figure(figsize=(16, 8))
sns.heatmap(correlation_matrix, annot=annotated_corr_matrix, fmt="", cmap="Reds", cbar=True, mask=mask,
            annot_kws={"size": 15})
plt.title("Correlation matrix")
plt.show()

## Linear Regressions

In [ ]:
# Bereite die Daten vor: Entferne NaN-Werte
data_cleaned = data_cleaned.dropna(subset=["attractive", "credebility", "ai" ,"purchase", "attitude"])

# Definiere die unabhängige Variable (Q15_mean) und die abhängige Variable (Q13_mean)
X_1 = data_cleaned[["attractive", "credebility", "ai"]]  # Unabhängige Variable (Q15)
y_1 = data_cleaned["purchase"]  # Abhängige Variable (Q13)

# Füge einen konstanten Term (Intercept) zur unabhängigen Variable hinzu
X_1 = sm.add_constant(X_1)

# Erstelle das Regressionsmodell
model = sm.OLS(y_1, X_1).fit(cov_type="HC3")

# Zeige die Zusammenfassung des Modells an
print(model.summary())

### Attractivenes -> Purchase interest

In [ ]:
# Konvertiere die Spalten in numerische Werte, erzwinge dabei NaN für nicht-numerische Werte
data_cleaned["Q15_stdev"] = np.std(data_cleaned[["Q15_1", "Q15_2", "Q15_3"]],axis=1, ddof=1)

# Jetzt gruppiere und berechne den Durchschnitt
result_mean_1 = data_cleaned.groupby("attractive")["purchase"].mean()
result_stdev_1 = data_cleaned.groupby("attractive")["purchase"].std(ddof=1)

# Zeige die Ergebnisse
print(data_cleaned["purchase"].mean())
print("mean:")
print(result_mean_1)
print("")
print("stdev:")
print(result_stdev_1)

In [ ]:
# Daten aufteilen
group_true_1 = data_cleaned[data_cleaned["attractive"] == True]["purchase"].dropna()
group_false_1 = data_cleaned[data_cleaned["attractive"] == False]["purchase"].dropna()

# Mann-Whitney-U-Test
u_stat_1, p_value_1 = mannwhitneyu(group_true_1, group_false_1, alternative='two-sided')

print(f"U-Statistik: {u_stat_1}")
print(f"P-Wert: {p_value_1}")

if p_value_1 < 0.05:
    print("Significant difference between True and False (p < 0.05).")
else:
    print("no sifnificant difference (p >= 0.05).")

### credebility -> purchase interest

In [ ]:
# Definiere die unabhängige Variable (Q15_mean) und die abhängige Variable (Q13_mean)
X_2 = data_cleaned[["credebility"]]
y_2 = data_cleaned["purchase"]

# Füge einen konstanten Term (Intercept) zur unabhängigen Variable hinzu
X_2 = sm.add_constant(X_2)

# Erstelle das Regressionsmodell
model = sm.OLS(y_2, X_2).fit(cov_type="HC3")

# Zeige die Zusammenfassung des Modells an
print(model.summary())

### ai -> purchase interest

In [ ]:
result_mean_2 = data_cleaned.groupby("ai")["purchase"].mean()
result_stdev_2 = data_cleaned.groupby("ai")["purchase"].std(ddof=1)

# Zeige die Ergebnisse
print(data_cleaned["purchase"].mean())
print("mean:")
print(result_mean_2)
print("")
print("stdev:")
print(result_stdev_2)

In [ ]:
group_true_2 = data_cleaned[data_cleaned["ai"] == True]["purchase"].dropna()
group_false_2 = data_cleaned[data_cleaned["ai"] == False]["purchase"].dropna()

# Mann-Whitney-U-Test
u_stat_2, p_value_2 = mannwhitneyu(group_true_2, group_false_2, alternative='two-sided')

print(f"U-Statistik: {u_stat_2}")
print(f"P-Wert: {p_value_2}")

if p_value_2 < 0.05:
    print("Significant difference between True and False (p < 0.05).")
else:
    print("no sifnificant difference (p >= 0.05).")

### Mediator (credebility): attractive -> credebil -> purchase interest

In [ ]:
# Regression 1: Credibility (Z) auf purchase (Y)
X1 = sm.add_constant(data_cleaned["attractive"])
model1 = sm.OLS(data_cleaned["purchase"], X1).fit(cov_type="HC3")
print("Regression 1: attractive -> purchase")
print(model1.summary())

# Regression 2: Credibility (Z) auf credebility (M)
X2 = sm.add_constant(data_cleaned["attractive"])
model2 = sm.OLS(data_cleaned["credebility"], X2).fit(cov_type="HC3")
print("\nRegression 2: attractive -> credebility")
print(model2.summary())

# Regression 3: Credibility (Z) + credebility (M) auf purchase (Y)
X3 = sm.add_constant(data_cleaned[["attractive", "credebility"]])
model3 = sm.OLS(data_cleaned["purchase"], X3).fit(cov_type="HC3")
print("\nRegression 3: attractive + credebility -> purchase")
print(model3.summary())

# Berechnung des indirekten Effekts
indirect_effect = model2.params["attractive"] * model3.params["credebility"]
direct_effect = model3.params["attractive"]
total_effect = model1.params["attractive"]

print("\nIndirekter Effekt (attractive -> credebility -> purchase):", indirect_effect)
print("Direkter Effekt (attractive -> purchase):", direct_effect)
print("Totaler Effekt (attractive -> purchase):", total_effect)

# Bootstrap-Test für den indirekten Effekt
results = pg.mediation_analysis(
    data=data_cleaned,
    x="attractive",
    m="credebility",
    y="purchase",
    alpha=0.05,
    n_boot=5000,
    seed=2026,
)
print("\nBootstrap-Mediation-Analyse:")
print(results)

### moderating effect ai on credebility -> purchase interest

In [ ]:
# Include the group main effect and interaction, allowing group intercepts to differ.
moderation_data = data_cleaned.assign(
    cred_x_ai=data_cleaned["credebility"] * data_cleaned["ai"]
)
moderation_model = sm.OLS(
    moderation_data["purchase"],
    sm.add_constant(moderation_data[["credebility", "ai", "cred_x_ai"]])
).fit(cov_type="HC3")
print(moderation_model.summary())


## attitude

### Joint linear model on attitude towards influencer

In [ ]:
# Bereite die Daten vor: Entferne NaN-Werte
data_cleaned = data_cleaned.dropna(subset=["attractive", "credebility", "ai" ,"attitude"])

# Definiere die unabhängige Variable (Q15_mean) und die abhängige Variable (Q13_mean)
X_4 = data_cleaned[["attractive", "credebility", "ai"]]  # Unabhängige Variable (Q15)
y_4 = data_cleaned["attitude"]  # Abhängige Variable (Q13)

# Füge einen konstanten Term (Intercept) zur unabhängigen Variable hinzu
X_4 = sm.add_constant(X_4)

# Erstelle das Regressionsmodell
model = sm.OLS(y_4, X_4).fit(cov_type="HC3")

# Zeige die Zusammenfassung des Modells an
print(model.summary())

### Attractivness -> Attitude towards influencer

In [ ]:
data_cleaned["Q14_stdev"] = np.std(data_cleaned[["Q14_1", "Q14_2", "Q14_3"]],axis=1, ddof=1)

# Jetzt gruppiere und berechne den Durchschnitt
result_mean_4 = data_cleaned.groupby("attractive")["attitude"].mean()
result_stdev_4 = data_cleaned.groupby("attractive")["attitude"].std(ddof=1)

# Zeige die Ergebnisse
print(data_cleaned["attitude"].mean())
print("mean:")
print(result_mean_4)
print("")
print("stdev:")
print(result_stdev_4)

In [ ]:
# Daten aufteilen
group_true_5 = data_cleaned[data_cleaned["attractive"] == True]["attitude"].dropna()
group_false_5 = data_cleaned[data_cleaned["attractive"] == False]["attitude"].dropna()

# Mann-Whitney-U-Test
u_stat_5, p_value_5 = mannwhitneyu(group_true_5, group_false_5, alternative='two-sided')

print(f"U-Statistik: {u_stat_5}")
print(f"P-Wert: {p_value_5}")

if p_value_5 < 0.05:
    print("Significant difference between True and False (p < 0.05).")
else:
    print("no sifnificant difference (p >= 0.05).")

### credebility -> attitude

In [ ]:
# Definiere die unabhängige Variable (Q15_mean) und die abhängige Variable (Q13_mean)
X_6 = data_cleaned[["credebility"]]
y_6 = data_cleaned["attitude"]

# Füge einen konstanten Term (Intercept) zur unabhängigen Variable hinzu
X_6 = sm.add_constant(X_6)

# Erstelle das Regressionsmodell
model = sm.OLS(y_6, X_6).fit(cov_type="HC3")

# Zeige die Zusammenfassung des Modells an
print(model.summary())

In [ ]:
result_mean_7 = data_cleaned.groupby("ai")["attitude"].mean()
result_stdev_7 = data_cleaned.groupby("ai")["attitude"].std(ddof=1)

# Zeige die Ergebnisse
print(data_cleaned["attitude"].mean())
print("mean:")
print(result_mean_7)
print("")
print("stdev:")
print(result_stdev_7)

In [ ]:
group_true_7 = data_cleaned[data_cleaned["ai"] == True]["attitude"].dropna()
group_false_7 = data_cleaned[data_cleaned["ai"] == False]["attitude"].dropna()

# Mann-Whitney-U-Test
u_stat_7, p_value_7 = mannwhitneyu(group_true_7, group_false_7, alternative='two-sided')

print(f"U-Statistik: {u_stat_7}")
print(f"P-Wert: {p_value_7}")

if p_value_7 < 0.05:
    print("Significant difference between True and False (p < 0.05).")
else:
    print("no sifnificant difference (p >= 0.05).")

### Mediator

In [ ]:
# Regression 1: Credibility (Z) auf attitude (Y)
X1 = sm.add_constant(data_cleaned["attractive"])
model1 = sm.OLS(data_cleaned["attitude"], X1).fit(cov_type="HC3")
print("Regression 1: attractive -> attitude")
print(model1.summary())

# Regression 2: Credibility (Z) auf credebility (M)
X2 = sm.add_constant(data_cleaned["attractive"])
model2 = sm.OLS(data_cleaned["credebility"], X2).fit(cov_type="HC3")
print("\nRegression 2: attractive -> credebility")
print(model2.summary())

# Regression 3: Credibility (Z) + credebility (M) auf attitude (Y)
X3 = sm.add_constant(data_cleaned[["attractive", "credebility"]])
model3 = sm.OLS(data_cleaned["attitude"], X3).fit(cov_type="HC3")
print("\nRegression 3: attractive + credebility -> attitude")
print(model3.summary())

# Berechnung des indirekten Effekts
indirect_effect = model2.params["attractive"] * model3.params["credebility"]
direct_effect = model3.params["attractive"]
total_effect = model1.params["attractive"]

print("\nIndirekter Effekt (attractive -> credebility -> attitude):", indirect_effect)
print("Direkter Effekt (attractive -> attitude):", direct_effect)
print("Totaler Effekt (attractive -> attitude):", total_effect)

# Bootstrap-Test für den indirekten Effekt
results = pg.mediation_analysis(
    data=data_cleaned,
    x="attractive",
    m="credebility",
    y="attitude",
    alpha=0.05,
    n_boot=5000,
    seed=2026,
)
print("\nBootstrap-Mediation-Analyse:")
print(results)

### Moderator AI

In [ ]:
# Include the group main effect and interaction, allowing group intercepts to differ.
moderation_data = data_cleaned.assign(
    cred_x_ai=data_cleaned["credebility"] * data_cleaned["ai"]
)
moderation_model = sm.OLS(
    moderation_data["attitude"],
    sm.add_constant(moderation_data[["credebility", "ai", "cred_x_ai"]])
).fit(cov_type="HC3")
print(moderation_model.summary())


## Attitude -> purchase

In [ ]:
X_10 = data_cleaned[["attitude"]]
y_10 = data_cleaned["purchase"]

# Füge einen konstanten Term (Intercept) zur unabhängigen Variable hinzu
X_10 = sm.add_constant(X_10)

# Erstelle das Regressionsmodell
model = sm.OLS(y_10, X_10).fit(cov_type="HC3")

# Zeige die Zusammenfassung des Modells an
print(model.summary())

## credebility -> attitude -> purchase

In [ ]:
# Regression 1: Credibility (Z) auf Purchase (Y)
X1 = sm.add_constant(data_cleaned["credebility"])
model1 = sm.OLS(data_cleaned["purchase"], X1).fit(cov_type="HC3")
print("Regression 1: Credibility -> Purchase")
print(model1.summary())

# Regression 2: Credibility (Z) auf Attitude (M)
X2 = sm.add_constant(data_cleaned["credebility"])
model2 = sm.OLS(data_cleaned["attitude"], X2).fit(cov_type="HC3")
print("\nRegression 2: Credibility -> Attitude")
print(model2.summary())

# Regression 3: Credibility (Z) + Attitude (M) auf Purchase (Y)
X3 = sm.add_constant(data_cleaned[["credebility", "attitude"]])
model3 = sm.OLS(data_cleaned["purchase"], X3).fit(cov_type="HC3")
print("\nRegression 3: Credibility + Attitude -> Purchase")
print(model3.summary())

# Berechnung des indirekten Effekts
indirect_effect = model2.params["credebility"] * model3.params["attitude"]
direct_effect = model3.params["credebility"]
total_effect = model1.params["credebility"]

print("\nIndirekter Effekt (Credibility -> Attitude -> Purchase):", indirect_effect)
print("Direkter Effekt (Credibility -> Purchase):", direct_effect)
print("Totaler Effekt (Credibility -> Purchase):", total_effect)

# Bootstrap-Test für den indirekten Effekt
results = pg.mediation_analysis(
    data=data_cleaned,
    x="credebility",
    m="attitude",
    y="purchase",
    alpha=0.05,
    n_boot=5000,
    seed=2026,
)
print("\nBootstrap-Mediation-Analyse:")
print(results)

## attractive → attitude → purchase

In [ ]:
# Regression 1: attractive (Z) auf Purchase (Y)
X1 = sm.add_constant(data_cleaned["attractive"])
model1 = sm.OLS(data_cleaned["purchase"], X1).fit(cov_type="HC3")
print("Regression 1: attractive -> Purchase")
print(model1.summary())

# Regression 2: Credibility (Z) auf Attitude (M)
X2 = sm.add_constant(data_cleaned["attractive"])
model2 = sm.OLS(data_cleaned["attitude"], X2).fit(cov_type="HC3")
print("\nRegression 2: attractive -> Attitude")
print(model2.summary())

# Regression 3: Credibility (Z) + Attitude (M) auf Purchase (Y)
X3 = sm.add_constant(data_cleaned[["attractive", "attitude"]])
model3 = sm.OLS(data_cleaned["purchase"], X3).fit(cov_type="HC3")
print("\nRegression 3: attractive + Attitude -> Purchase")
print(model3.summary())

# Berechnung des indirekten Effekts
indirect_effect = model2.params["attractive"] * model3.params["attitude"]
direct_effect = model3.params["attractive"]
total_effect = model1.params["attractive"]

print("\nIndirekter Effekt (attractive -> Attitude -> Purchase):", indirect_effect)
print("Direkter Effekt (attractive -> Purchase):", direct_effect)
print("Totaler Effekt (attractive-> Purchase):", total_effect)

# Bootstrap-Test für den indirekten Effekt
results = pg.mediation_analysis(
    data=data_cleaned,
    x="attractive",
    m="attitude",
    y="purchase",
    alpha=0.05,
    n_boot=5000,
    seed=2026,
)
print("\nBootstrap-Mediation-Analyse:")
print(results)

## moderator attractive

In [ ]:
data_cleaned["cred_x_attractive"] = data_cleaned["credebility"] * data_cleaned["attractive"]
X_9 = data_cleaned[["credebility", "cred_x_attractive" ,"attractive"]]  # Unabhängige Variable (Q15)
y_9 = data_cleaned["attitude"]  # Abhängige Variable (Q13)

# Füge einen konstanten Term (Intercept) zur unabhängigen Variable hinzu
X_9 = sm.add_constant(X_9)

# Erstelle das Regressionsmodell
model = sm.OLS(y_9, X_9).fit(cov_type="HC3")

# Zeige die Zusammenfassung des Modells an
print(model.summary())

## Sensitivity to excluding failed AI-label recognition

Repeat the joint outcome regressions, interaction models, and indirect-effect
analyses among label recognizers. This is a selected post-exposure subgroup.
Compare it with the primary eligible sample. Treat additional unplanned analyses
as exploratory. The original pairwise tests and correlations are unadjusted.


In [ ]:
sensitivity = recognition_sensitivity.copy()
for column in ["attractive", "ai"]:
    sensitivity[column] = sensitivity[column].astype(int)
for scale, items in {
    "credebility": ["Q13_1", "Q13_2", "Q13_4"],
    "attitude": ["Q14_1", "Q14_2", "Q14_3"],
    "purchase": ["Q15_1", "Q15_2", "Q15_3"],
}.items():
    sensitivity[scale] = sensitivity[items].apply(
        pd.to_numeric, errors="coerce"
    ).mean(axis=1, skipna=False)
sensitivity = sensitivity.dropna(
    subset=["attractive", "ai", "credebility", "attitude", "purchase"]
)
print("Complete-case sensitivity sample:", len(sensitivity))
sensitivity["cred_x_ai"] = sensitivity["credebility"] * sensitivity["ai"]
for outcome in ["purchase", "attitude"]:
    print("Sensitivity outcome:", outcome)
    for predictors in [
        ["attractive", "credebility", "ai"],
        ["credebility", "ai", "cred_x_ai"],
    ]:
        print(sm.OLS(
            sensitivity[outcome], sm.add_constant(sensitivity[predictors])
        ).fit(cov_type="HC3").summary())
for predictor, mediator, outcome in [
    ("attractive", "credebility", "purchase"),
    ("attractive", "credebility", "attitude"),
    ("credebility", "attitude", "purchase"),
    ("attractive", "attitude", "purchase"),
]:
    print(predictor, mediator, outcome)
    display(pg.mediation_analysis(
        data=sensitivity, x=predictor, m=mediator, y=outcome,
        n_boot=5000, seed=2026
    ))
